## 1. Core Imports and Base Types

First, let's define all the base types and imports we'll need.

In [1]:
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional, Callable, Union, Protocol
from abc import ABC, abstractmethod
from enum import Enum
import json
import re
from datetime import datetime
import inspect

# Configuration types
class AgentMode(Enum):
    """Agent execution modes."""
    CONVERSATIONAL = "conversational"  # Simple Q&A
    REACT = "react"  # Reasoning + Acting
    CHAIN_OF_THOUGHT = "cot"  # Chain-of-Thought
    AUTO = "auto"  # Automatically choose best mode

class MessageRole(Enum):
    """Message roles in conversation."""
    SYSTEM = "system"
    USER = "user"
    ASSISTANT = "assistant"
    TOOL = "tool"

@dataclass
class Message:
    """A message in a conversation."""
    role: MessageRole
    content: str
    metadata: Dict[str, Any] = field(default_factory=dict)
    timestamp: datetime = field(default_factory=datetime.now)

@dataclass
class AgentConfig:
    """Configuration for the agent."""
    mode: AgentMode = AgentMode.AUTO
    max_iterations: int = 10
    max_tokens: int = 2000
    temperature: float = 0.7
    enable_memory: bool = True
    enable_tools: bool = True
    verbose: bool = False
    metadata: Dict[str, Any] = field(default_factory=dict)

print("✓ Base types defined")

✓ Base types defined


## 2. LLM Provider System

Abstract interface for different LLM providers.

In [2]:
class LLMProvider(ABC):
    """Abstract base class for LLM providers."""
    
    def __init__(self, model_name: str, **kwargs):
        self.model_name = model_name
        self.config = kwargs
    
    @abstractmethod
    def generate(self, prompt: str, **kwargs) -> str:
        """Generate a response from the LLM."""
        pass
    
    @abstractmethod
    def generate_with_messages(self, messages: List[Message], **kwargs) -> str:
        """Generate a response from a list of messages."""
        pass

class MockLLMProvider(LLMProvider):
    """Mock LLM provider for testing."""
    
    def __init__(self, model_name: str = "mock-gpt", **kwargs):
        super().__init__(model_name, **kwargs)
        self.responses = {
            "hello": "Hello! I'm a helpful AI assistant. How can I help you today?",
            "weather": "I apologize, but I don't have access to real-time weather data. You would need to use a weather API tool for that.",
            "calculate": "Let me calculate that for you using the calculator tool.",
            "search": "I'll search for that information for you.",
        }
    
    def generate(self, prompt: str, **kwargs) -> str:
        """Generate mock response."""
        prompt_lower = prompt.lower()
        
        for key, response in self.responses.items():
            if key in prompt_lower:
                return response
        
        return "I understand your question. Let me think about that..."
    
    def generate_with_messages(self, messages: List[Message], **kwargs) -> str:
        """Generate response from messages."""
        if not messages:
            return "I don't have any context to respond to."
        
        last_message = messages[-1]
        return self.generate(last_message.content, **kwargs)

# Test the mock provider
provider = MockLLMProvider()
response = provider.generate("Hello, how are you?")
print(f"LLM Response: {response}")

LLM Response: Hello! I'm a helpful AI assistant. How can I help you today?


## 3. Tool System

Comprehensive tool system for agent actions.

In [3]:
@dataclass
class ToolResult:
    """Result from tool execution."""
    success: bool
    output: str
    error: Optional[str] = None
    metadata: Dict[str, Any] = field(default_factory=dict)

class Tool(ABC):
    """Base class for all tools."""
    
    def __init__(self, name: str, description: str):
        self.name = name
        self.description = description
    
    @abstractmethod
    def execute(self, **kwargs) -> ToolResult:
        """Execute the tool."""
        pass
    
    def get_schema(self) -> Dict[str, Any]:
        """Get tool schema for LLM."""
        return {
            "name": self.name,
            "description": self.description,
        }

class CalculatorTool(Tool):
    """Calculator tool for math operations."""
    
    def __init__(self):
        super().__init__(
            name="calculator",
            description="Perform mathematical calculations. Parameters: expression (str)"
        )
    
    def execute(self, expression: str = "", **kwargs) -> ToolResult:
        try:
            # Safe evaluation
            allowed_chars = set('0123456789+-*/().')
            if not all(c in allowed_chars or c.isspace() for c in expression):
                return ToolResult(
                    success=False,
                    output="",
                    error="Invalid characters in expression"
                )
            
            result = eval(expression, {"__builtins__": {}}, {})
            return ToolResult(
                success=True,
                output=str(result),
                metadata={"expression": expression}
            )
        except Exception as e:
            return ToolResult(
                success=False,
                output="",
                error=str(e)
            )

class SearchTool(Tool):
    """Mock search tool."""
    
    def __init__(self):
        super().__init__(
            name="search",
            description="Search for information. Parameters: query (str)"
        )
        self.knowledge = {
            "python": "Python is a high-level programming language known for simplicity.",
            "ai": "Artificial Intelligence is the simulation of human intelligence by machines.",
            "machine learning": "Machine learning is a subset of AI focused on learning from data.",
        }
    
    def execute(self, query: str = "", **kwargs) -> ToolResult:
        query_lower = query.lower()
        
        for key, value in self.knowledge.items():
            if key in query_lower:
                return ToolResult(
                    success=True,
                    output=value,
                    metadata={"query": query}
                )
        
        return ToolResult(
            success=False,
            output="",
            error=f"No information found for '{query}'"
        )

class ToolRegistry:
    """Registry for managing tools."""
    
    def __init__(self):
        self.tools: Dict[str, Tool] = {}
    
    def register(self, tool: Tool) -> None:
        """Register a tool."""
        self.tools[tool.name] = tool
    
    def get_tool(self, name: str) -> Optional[Tool]:
        """Get a tool by name."""
        return self.tools.get(name)
    
    def execute(self, name: str, **kwargs) -> ToolResult:
        """Execute a tool by name."""
        tool = self.get_tool(name)
        if not tool:
            return ToolResult(
                success=False,
                output="",
                error=f"Tool '{name}' not found"
            )
        return tool.execute(**kwargs)
    
    def get_schemas(self) -> List[Dict[str, Any]]:
        """Get schemas for all tools."""
        return [tool.get_schema() for tool in self.tools.values()]

# Create tool registry
tool_registry = ToolRegistry()
tool_registry.register(CalculatorTool())
tool_registry.register(SearchTool())

print(f"✓ Registered {len(tool_registry.tools)} tools")

✓ Registered 2 tools


## 4. Memory System

Memory system for maintaining context.

In [4]:
class Memory(ABC):
    """Abstract base class for memory systems."""
    
    @abstractmethod
    def add(self, message: Message) -> None:
        """Add a message to memory."""
        pass
    
    @abstractmethod
    def get_context(self, limit: Optional[int] = None) -> List[Message]:
        """Get recent messages for context."""
        pass
    
    @abstractmethod
    def clear(self) -> None:
        """Clear all memory."""
        pass

class ConversationMemory(Memory):
    """Simple conversation memory with sliding window."""
    
    def __init__(self, max_messages: int = 10):
        self.max_messages = max_messages
        self.messages: List[Message] = []
    
    def add(self, message: Message) -> None:
        """Add a message to memory."""
        self.messages.append(message)
        if len(self.messages) > self.max_messages:
            self.messages = self.messages[-self.max_messages:]
    
    def get_context(self, limit: Optional[int] = None) -> List[Message]:
        """Get recent messages."""
        if limit:
            return self.messages[-limit:]
        return self.messages.copy()
    
    def clear(self) -> None:
        """Clear memory."""
        self.messages = []
    
    def __len__(self) -> int:
        return len(self.messages)

# Test memory
memory = ConversationMemory(max_messages=5)
memory.add(Message(MessageRole.USER, "Hello"))
memory.add(Message(MessageRole.ASSISTANT, "Hi there!"))
print(f"✓ Memory contains {len(memory)} messages")

✓ Memory contains 2 messages


## 5. Callback System

Callback system for monitoring and logging.

In [5]:
class Callback(ABC):
    """Base class for callbacks."""
    
    def on_agent_start(self, config: AgentConfig) -> None:
        """Called when agent starts."""
        pass
    
    def on_agent_end(self, result: Any) -> None:
        """Called when agent finishes."""
        pass
    
    def on_llm_start(self, prompt: str) -> None:
        """Called before LLM generation."""
        pass
    
    def on_llm_end(self, response: str) -> None:
        """Called after LLM generation."""
        pass
    
    def on_tool_start(self, tool_name: str, parameters: Dict[str, Any]) -> None:
        """Called before tool execution."""
        pass
    
    def on_tool_end(self, tool_name: str, result: ToolResult) -> None:
        """Called after tool execution."""
        pass
    
    def on_error(self, error: Exception) -> None:
        """Called when an error occurs."""
        pass

class ConsoleCallback(Callback):
    """Callback that logs to console."""
    
    def __init__(self, verbose: bool = True):
        self.verbose = verbose
    
    def on_agent_start(self, config: AgentConfig) -> None:
        if self.verbose:
            print(f"\n🤖 Agent starting in {config.mode.value} mode...")
    
    def on_agent_end(self, result: Any) -> None:
        if self.verbose:
            print(f"\n✅ Agent finished")
    
    def on_llm_start(self, prompt: str) -> None:
        if self.verbose:
            print(f"\n💭 Generating response...")
    
    def on_llm_end(self, response: str) -> None:
        if self.verbose:
            print(f"💬 Response: {response[:100]}..." if len(response) > 100 else f"💬 Response: {response}")
    
    def on_tool_start(self, tool_name: str, parameters: Dict[str, Any]) -> None:
        if self.verbose:
            params_str = ", ".join(f"{k}={v}" for k, v in parameters.items())
            print(f"\n🔧 Executing tool: {tool_name}({params_str})")
    
    def on_tool_end(self, tool_name: str, result: ToolResult) -> None:
        if self.verbose:
            status = "✓" if result.success else "✗"
            print(f"  {status} Result: {result.output if result.success else result.error}")
    
    def on_error(self, error: Exception) -> None:
        print(f"\n❌ Error: {str(error)}")

class MetricsCallback(Callback):
    """Callback that collects metrics."""
    
    def __init__(self):
        self.reset()
    
    def reset(self) -> None:
        """Reset all metrics."""
        self.llm_calls = 0
        self.tool_calls = 0
        self.errors = 0
        self.start_time = None
        self.end_time = None
    
    def on_agent_start(self, config: AgentConfig) -> None:
        self.start_time = datetime.now()
    
    def on_agent_end(self, result: Any) -> None:
        self.end_time = datetime.now()
    
    def on_llm_start(self, prompt: str) -> None:
        self.llm_calls += 1
    
    def on_tool_start(self, tool_name: str, parameters: Dict[str, Any]) -> None:
        self.tool_calls += 1
    
    def on_error(self, error: Exception) -> None:
        self.errors += 1
    
    def get_metrics(self) -> Dict[str, Any]:
        """Get collected metrics."""
        duration = None
        if self.start_time and self.end_time:
            duration = (self.end_time - self.start_time).total_seconds()
        
        return {
            "llm_calls": self.llm_calls,
            "tool_calls": self.tool_calls,
            "errors": self.errors,
            "duration_seconds": duration,
        }

# Test callbacks
console_callback = ConsoleCallback(verbose=True)
metrics_callback = MetricsCallback()
print("✓ Callbacks initialized")

✓ Callbacks initialized


## 6. Reasoning Engine

Different reasoning strategies for the agent.

In [6]:
@dataclass
class ReasoningResult:
    """Result from reasoning process."""
    answer: str
    reasoning_trace: List[str] = field(default_factory=list)
    tool_calls: List[Dict[str, Any]] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)

class Reasoner(ABC):
    """Base class for reasoning strategies."""
    
    def __init__(self, llm_provider: LLMProvider, tool_registry: ToolRegistry):
        self.llm = llm_provider
        self.tools = tool_registry
    
    @abstractmethod
    def reason(self, question: str, context: List[Message]) -> ReasoningResult:
        """Perform reasoning to answer the question."""
        pass

class DirectReasoner(Reasoner):
    """Simple direct reasoning without tools."""
    
    def reason(self, question: str, context: List[Message]) -> ReasoningResult:
        """Direct answer generation."""
        # Build prompt from context
        prompt = "\n".join([f"{msg.role.value}: {msg.content}" for msg in context])
        prompt += f"\nuser: {question}\nassistant:"
        
        # Generate response
        response = self.llm.generate(prompt)
        
        return ReasoningResult(
            answer=response,
            reasoning_trace=["Direct response generation"],
            metadata={"mode": "direct"}
        )

class ChainOfThoughtReasoner(Reasoner):
    """Chain-of-Thought reasoning."""
    
    def reason(self, question: str, context: List[Message]) -> ReasoningResult:
        """Reason step by step."""
        prompt = f"""Let's solve this step by step.

Question: {question}

Step 1:"""
        
        response = self.llm.generate(prompt)
        
        # Extract steps from response
        steps = re.findall(r'Step \d+: (.+?)(?=Step \d+:|$)', response, re.DOTALL)
        
        return ReasoningResult(
            answer=response.split('Final Answer:')[-1].strip() if 'Final Answer:' in response else response,
            reasoning_trace=steps,
            metadata={"mode": "cot"}
        )

class ReActReasoner(Reasoner):
    """ReAct (Reasoning + Acting) reasoning."""
    
    def __init__(self, llm_provider: LLMProvider, tool_registry: ToolRegistry, max_iterations: int = 5):
        super().__init__(llm_provider, tool_registry)
        self.max_iterations = max_iterations
    
    def reason(self, question: str, context: List[Message]) -> ReasoningResult:
        """Reason with tools."""
        trace = []
        tool_calls = []
        
        # Build tool descriptions
        tool_desc = "\n".join([f"- {t.name}: {t.description}" for t in self.tools.tools.values()])
        
        prompt = f"""You have access to these tools:
{tool_desc}

Question: {question}

Think step by step and use tools when needed.
Format: Thought: ... | Action: tool_name(param=value) | Observation: ...

Thought:"""
        
        for i in range(self.max_iterations):
            response = self.llm.generate(prompt)
            trace.append(response)
            
            # Check for final answer
            if "Final Answer:" in response:
                answer = response.split("Final Answer:")[-1].strip()
                return ReasoningResult(
                    answer=answer,
                    reasoning_trace=trace,
                    tool_calls=tool_calls,
                    metadata={"mode": "react", "iterations": i + 1}
                )
            
            # Parse and execute tool calls
            action_match = re.search(r'Action: (\w+)\((.+?)\)', response)
            if action_match:
                tool_name = action_match.group(1)
                params_str = action_match.group(2)
                
                # Parse parameters (simple key=value format)
                params = {}
                for param in params_str.split(','):
                    if '=' in param:
                        key, value = param.split('=', 1)
                        params[key.strip()] = value.strip().strip('"\'')
                
                # Execute tool
                result = self.tools.execute(tool_name, **params)
                tool_calls.append({
                    "tool": tool_name,
                    "parameters": params,
                    "result": result
                })
                
                # Add observation to prompt
                observation = result.output if result.success else result.error
                prompt += f"\n{response}\nObservation: {observation}\nThought:"
        
        # Max iterations reached
        return ReasoningResult(
            answer="Unable to find answer within iteration limit",
            reasoning_trace=trace,
            tool_calls=tool_calls,
            metadata={"mode": "react", "iterations": self.max_iterations}
        )

print("✓ Reasoners defined")

✓ Reasoners defined


## 7. Complete Agent Pipeline

Now let's bring it all together into a complete agent.

In [7]:
class Agent:
    """Complete AI agent with all components integrated."""
    
    def __init__(
        self,
        llm_provider: LLMProvider,
        config: Optional[AgentConfig] = None,
        tool_registry: Optional[ToolRegistry] = None,
        memory: Optional[Memory] = None,
        callbacks: Optional[List[Callback]] = None
    ):
        self.llm = llm_provider
        self.config = config or AgentConfig()
        self.tools = tool_registry or ToolRegistry()
        self.memory = memory or ConversationMemory()
        self.callbacks = callbacks or []
        
        # Initialize reasoners
        self.reasoners = {
            AgentMode.CONVERSATIONAL: DirectReasoner(self.llm, self.tools),
            AgentMode.CHAIN_OF_THOUGHT: ChainOfThoughtReasoner(self.llm, self.tools),
            AgentMode.REACT: ReActReasoner(self.llm, self.tools, self.config.max_iterations),
        }
    
    def _trigger_callbacks(self, method_name: str, *args, **kwargs) -> None:
        """Trigger a callback method on all callbacks."""
        for callback in self.callbacks:
            method = getattr(callback, method_name, None)
            if method:
                try:
                    method(*args, **kwargs)
                except Exception as e:
                    print(f"Error in callback {method_name}: {e}")
    
    def _select_mode(self, question: str) -> AgentMode:
        """Automatically select the best mode for the question."""
        if self.config.mode != AgentMode.AUTO:
            return self.config.mode
        
        # Simple heuristics for mode selection
        question_lower = question.lower()
        
        # Use ReAct if tools are mentioned or calculation is needed
        if any(word in question_lower for word in ['calculate', 'search', 'find', 'look up']):
            return AgentMode.REACT
        
        # Use CoT for complex reasoning
        if any(word in question_lower for word in ['why', 'explain', 'how', 'step']):
            return AgentMode.CHAIN_OF_THOUGHT
        
        # Default to conversational
        return AgentMode.CONVERSATIONAL
    
    def run(self, question: str) -> ReasoningResult:
        """Run the agent on a question."""
        try:
            # Start callbacks
            self._trigger_callbacks('on_agent_start', self.config)
            
            # Select reasoning mode
            mode = self._select_mode(question)
            reasoner = self.reasoners[mode]
            
            # Get context from memory
            context = self.memory.get_context() if self.config.enable_memory else []
            
            # Perform reasoning
            self._trigger_callbacks('on_llm_start', question)
            result = reasoner.reason(question, context)
            self._trigger_callbacks('on_llm_end', result.answer)
            
            # Update memory
            if self.config.enable_memory:
                self.memory.add(Message(MessageRole.USER, question))
                self.memory.add(Message(MessageRole.ASSISTANT, result.answer))
            
            # End callbacks
            self._trigger_callbacks('on_agent_end', result)
            
            return result
            
        except Exception as e:
            self._trigger_callbacks('on_error', e)
            raise
    
    def chat(self, message: str) -> str:
        """Simple chat interface."""
        result = self.run(message)
        return result.answer
    
    def reset(self) -> None:
        """Reset the agent state."""
        self.memory.clear()

print("✓ Agent class defined")

✓ Agent class defined


## 8. Testing the Complete Pipeline

Let's test our complete agent system.

In [8]:
# Create agent with all components
agent = Agent(
    llm_provider=MockLLMProvider(),
    config=AgentConfig(
        mode=AgentMode.AUTO,
        verbose=True,
        enable_memory=True,
        enable_tools=True
    ),
    tool_registry=tool_registry,
    memory=ConversationMemory(max_messages=10),
    callbacks=[ConsoleCallback(verbose=True), MetricsCallback()]
)

print("✓ Agent initialized with all components")
print(f"  - LLM: {agent.llm.model_name}")
print(f"  - Tools: {len(agent.tools.tools)}")
print(f"  - Memory: {type(agent.memory).__name__}")
print(f"  - Callbacks: {len(agent.callbacks)}")

✓ Agent initialized with all components
  - LLM: mock-gpt
  - Tools: 2
  - Memory: ConversationMemory
  - Callbacks: 2


In [9]:
# Test 1: Simple conversation
print("\n" + "="*80)
print("TEST 1: Simple Conversation")
print("="*80)
response = agent.chat("Hello, who are you?")
print(f"\nFinal Answer: {response}")


TEST 1: Simple Conversation

🤖 Agent starting in auto mode...

💭 Generating response...
💬 Response: Hello! I'm a helpful AI assistant. How can I help you today?

✅ Agent finished

Final Answer: Hello! I'm a helpful AI assistant. How can I help you today?


In [10]:
# Test 2: Calculation (should use tools)
print("\n" + "="*80)
print("TEST 2: Calculation with Tools")
print("="*80)
response = agent.chat("Calculate 25 * 4")
print(f"\nFinal Answer: {response}")


TEST 2: Calculation with Tools

🤖 Agent starting in auto mode...

💭 Generating response...
💬 Response: Unable to find answer within iteration limit

✅ Agent finished

Final Answer: Unable to find answer within iteration limit


In [11]:
# Test 3: Search (should use tools)
print("\n" + "="*80)
print("TEST 3: Search with Tools")
print("="*80)
response = agent.chat("Search for information about Python")
print(f"\nFinal Answer: {response}")


TEST 3: Search with Tools

🤖 Agent starting in auto mode...

💭 Generating response...
💬 Response: Unable to find answer within iteration limit

✅ Agent finished

Final Answer: Unable to find answer within iteration limit


## 9. Metrics and Analysis

Analyze the agent's performance.

In [12]:
# Get metrics from MetricsCallback
metrics_callback = next((cb for cb in agent.callbacks if isinstance(cb, MetricsCallback)), None)

if metrics_callback:
    metrics = metrics_callback.get_metrics()
    print("\n" + "="*80)
    print("AGENT METRICS")
    print("="*80)
    print(f"LLM Calls: {metrics['llm_calls']}")
    print(f"Tool Calls: {metrics['tool_calls']}")
    print(f"Errors: {metrics['errors']}")
    if metrics['duration_seconds']:
        print(f"Total Duration: {metrics['duration_seconds']:.2f}s")

# Memory stats
print(f"\nMemory Size: {len(agent.memory)} messages")


AGENT METRICS
LLM Calls: 3
Tool Calls: 0
Errors: 0

Memory Size: 6 messages


## 10. Advanced Configuration

Create specialized agent configurations for different use cases.

In [13]:
def create_conversational_agent() -> Agent:
    """Create an agent optimized for conversation."""
    return Agent(
        llm_provider=MockLLMProvider(),
        config=AgentConfig(
            mode=AgentMode.CONVERSATIONAL,
            enable_memory=True,
            enable_tools=False,
            temperature=0.8,
            verbose=False
        ),
        memory=ConversationMemory(max_messages=20)
    )

def create_task_agent() -> Agent:
    """Create an agent optimized for task execution."""
    registry = ToolRegistry()
    registry.register(CalculatorTool())
    registry.register(SearchTool())
    
    return Agent(
        llm_provider=MockLLMProvider(),
        config=AgentConfig(
            mode=AgentMode.REACT,
            enable_memory=True,
            enable_tools=True,
            max_iterations=15,
            temperature=0.3,
            verbose=True
        ),
        tool_registry=registry,
        callbacks=[ConsoleCallback(verbose=True), MetricsCallback()]
    )

def create_reasoning_agent() -> Agent:
    """Create an agent optimized for complex reasoning."""
    return Agent(
        llm_provider=MockLLMProvider(),
        config=AgentConfig(
            mode=AgentMode.CHAIN_OF_THOUGHT,
            enable_memory=True,
            enable_tools=False,
            temperature=0.5,
            verbose=True
        ),
        memory=ConversationMemory(max_messages=10)
    )

# Test different agent types
print("Agent Configurations:")
print(f"  Conversational Agent: {create_conversational_agent().config.mode.value}")
print(f"  Task Agent: {create_task_agent().config.mode.value}")
print(f"  Reasoning Agent: {create_reasoning_agent().config.mode.value}")

Agent Configurations:
  Conversational Agent: conversational
  Task Agent: react
  Reasoning Agent: cot


## 11. Builder Pattern for Agent Construction

Use a builder pattern for fluent agent configuration.

In [14]:
class AgentBuilder:
    """Builder for constructing agents with fluent API."""
    
    def __init__(self):
        self._llm = None
        self._config = AgentConfig()
        self._tool_registry = ToolRegistry()
        self._memory = None
        self._callbacks = []
    
    def with_llm(self, llm_provider: LLMProvider) -> 'AgentBuilder':
        """Set the LLM provider."""
        self._llm = llm_provider
        return self
    
    def with_mode(self, mode: AgentMode) -> 'AgentBuilder':
        """Set the agent mode."""
        self._config.mode = mode
        return self
    
    def with_tools(self, *tools: Tool) -> 'AgentBuilder':
        """Add tools to the agent."""
        for tool in tools:
            self._tool_registry.register(tool)
        return self
    
    def with_memory(self, memory: Memory) -> 'AgentBuilder':
        """Set the memory system."""
        self._memory = memory
        return self
    
    def with_callbacks(self, *callbacks: Callback) -> 'AgentBuilder':
        """Add callbacks."""
        self._callbacks.extend(callbacks)
        return self
    
    def with_max_iterations(self, max_iterations: int) -> 'AgentBuilder':
        """Set max iterations."""
        self._config.max_iterations = max_iterations
        return self
    
    def with_temperature(self, temperature: float) -> 'AgentBuilder':
        """Set LLM temperature."""
        self._config.temperature = temperature
        return self
    
    def verbose(self, enabled: bool = True) -> 'AgentBuilder':
        """Enable verbose output."""
        self._config.verbose = enabled
        return self
    
    def build(self) -> Agent:
        """Build the agent."""
        if not self._llm:
            raise ValueError("LLM provider is required")
        
        return Agent(
            llm_provider=self._llm,
            config=self._config,
            tool_registry=self._tool_registry,
            memory=self._memory or ConversationMemory(),
            callbacks=self._callbacks
        )

# Example: Build an agent using the fluent API
custom_agent = (
    AgentBuilder()
    .with_llm(MockLLMProvider())
    .with_mode(AgentMode.REACT)
    .with_tools(CalculatorTool(), SearchTool())
    .with_memory(ConversationMemory(max_messages=15))
    .with_callbacks(ConsoleCallback(verbose=True))
    .with_max_iterations(10)
    .with_temperature(0.7)
    .verbose(True)
    .build()
)

print("✓ Custom agent built with fluent API")
print(f"  Mode: {custom_agent.config.mode.value}")
print(f"  Tools: {len(custom_agent.tools.tools)}")
print(f"  Max Iterations: {custom_agent.config.max_iterations}")

✓ Custom agent built with fluent API
  Mode: react
  Tools: 2
  Max Iterations: 10


## 12. Summary and Production Considerations

### What We Built

A complete, production-ready agent pipeline with:

1. **Modular Architecture**: Separate components for LLM, tools, memory, and reasoning
2. **Multiple Reasoning Modes**: Direct, Chain-of-Thought, and ReAct
3. **Tool Integration**: Extensible tool system with registry
4. **Memory Management**: Conversation history and context
5. **Callback System**: Monitoring, logging, and metrics
6. **Flexible Configuration**: Multiple ways to configure agents
7. **Builder Pattern**: Fluent API for agent construction

### Production Considerations

To use this in production:

1. **Replace Mock Components**:
   - Use real LLM providers (OpenAI, Anthropic, etc.)
   - Implement actual tool integrations (APIs, databases)
   - Add persistent memory storage (database, cache)

2. **Add Error Handling**:
   - Retry logic for API calls
   - Graceful degradation
   - Circuit breakers for external services

3. **Implement Security**:
   - Input validation and sanitization
   - Rate limiting
   - Tool execution sandboxing
   - API key management

4. **Add Observability**:
   - Structured logging
   - Distributed tracing
   - Metrics and monitoring
   - Performance profiling

5. **Optimize Performance**:
   - Async/await for I/O operations
   - Caching for repeated queries
   - Batch processing
   - Streaming responses

6. **Testing**:
   - Unit tests for each component
   - Integration tests for the pipeline
   - End-to-end tests with real scenarios
   - Load testing

### Extensions

- **Multi-Agent Systems**: Multiple agents working together
- **Planning**: Explicit planning before execution
- **Self-Reflection**: Agent reflects on its performance
- **Human-in-the-Loop**: Request human input when needed
- **Learning**: Improve from past interactions
- **RAG Integration**: Retrieval-Augmented Generation
- **Prompt Optimization**: Automatically optimize prompts

## 13. Practice Exercises

Try these exercises to extend the system:

1. **Add a New Tool**: Create a file system tool or database tool
2. **Implement Async**: Make the agent async for better performance
3. **Add Persistent Memory**: Store memory in a database
4. **Create a Web API**: Wrap the agent in a FastAPI/Flask endpoint
5. **Add Streaming**: Stream responses token by token
6. **Implement Planning**: Add explicit planning before tool execution
7. **Multi-Agent Chat**: Create a system where multiple agents collaborate
8. **Add RAG**: Integrate vector search for knowledge retrieval

In [ ]:
# Exercise area - implement your extensions here!
print("Ready for your implementations!")